# Lab 1 · Experiment tracking
**Block 1 · about 50 min.** StreamCo wants to predict which customers will cancel in the next 30 days.
Today every training attempt becomes an MLflow **run** inside the **experiment** `churn-workshop`.

Before running: in a terminal, from the workshop folder, start the tracking server
```
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
```
and open http://127.0.0.1:5000.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")          # churn_data.py, churn_model.py
warnings.filterwarnings("ignore")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")   # the server you started in the terminal
mlflow.set_experiment("churn-workshop")
print("MLflow", mlflow.__version__)

2026/11/04 20:40:33 INFO mlflow.tracking.fluent: Experiment with name 'churn-workshop' does not exist. Creating a new experiment.


MLflow 3.17.2


## Step 1 · Look at the data
Feb–Jun 2026 snapshots, one row per customer per month. We **split by time**: train on Feb–Apr,
validate on May–Jun. Never shuffle the future into training.

In [2]:
from churn_data import history, time_split, FEATURES, TARGET
df = history()
X_tr, y_tr, X_va, y_va = time_split(df)
print(df.shape, "| churn rate:", round(df[TARGET].mean(), 3))
df.head()

(25000, 10) | churn rate: 0.101


,CUSTOMER_ID,SNAPSHOT_TS,LOGINS_30D,PURCHASES_30D,SPEND_30D,TICKETS_30D,TENURE_DAYS,PLAN_TIER,COUNTRY,CHURNED
0,1,2026-02-01,19,4,175.17,2,351,0,DE,1
1,2,2026-02-01,13,3,22.43,2,110,2,DE,0
2,3,2026-02-01,26,0,0.00,0,983,1,ES,0
3,4,2026-02-01,27,3,240.80,1,1065,1,IT,0
4,5,2026-02-01,28,3,346.57,0,1196,1,DE,0


## Step 2 · Your first run: params, metrics, tags
A run records **parameters** (inputs you chose), **metrics** (numbers you measured) and **tags** (labels to search by).

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score
from churn_model import baseline

with mlflow.start_run(run_name="logreg_baseline") as run:
    model = baseline().fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]

    mlflow.log_params({"model": "LogisticRegression", "max_iter": 1000,
                       "train_period": "2026-02..2026-04", "valid_period": "2026-05..2026-06"})
    mlflow.log_metrics({"val_auc": roc_auc_score(y_va, proba),
                        "val_pr_auc": average_precision_score(y_va, proba)})
    mlflow.set_tags({"owner": "your-name", "stage": "baseline"})

print("run_id:", run.info.run_id)

🏃 View run logreg_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/fc650f6bdaae400284dea307161965ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
run_id: fc650f6bdaae400284dea307161965ab


👉 **In the UI:** open the experiment, click the run, find the params, metrics and tags.

## Step 3 · Artifacts and datasets
Artifacts are **files** attached to a run: plots, reports, feature lists.
Logging the **dataset** records which data (and which version of it) the model saw.

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

with mlflow.start_run(run_name="logreg_with_artifacts"):
    model = baseline().fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]
    mlflow.log_metric("val_auc", roc_auc_score(y_va, proba))

    # 1. Plots
    fig, ax = plt.subplots(figsize=(5, 4))
    RocCurveDisplay.from_predictions(y_va, proba, ax=ax)
    mlflow.log_figure(fig, "plots/roc_curve.png")
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_va, proba > 0.35, ax=ax)
    mlflow.log_figure(fig, "plots/confusion_matrix.png")
    plt.close("all")

    # 2. Structured files
    mlflow.log_dict({"features": FEATURES, "target": TARGET}, "config/features.json")
    mlflow.log_text("Threshold 0.35 chosen with the CRM team.", "notes.txt")

    # 3. Datasets: what the model was trained and validated on
    train_ds = mlflow.data.from_pandas(df[df.SNAPSHOT_TS < "2026-05-01"], name="churn_history",
                                       targets=TARGET)
    mlflow.log_input(train_ds, context="training")

🏃 View run logreg_with_artifacts at: http://127.0.0.1:5000/#/experiments/1/runs/1551e8e6bb344011a4a518c51b818cd7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


👉 **In the UI:** open the run → *Artifacts* tab for the plots, and the *Datasets* column on the runs list.

## Step 4 · Metrics over steps (a learning curve)
A metric logged with `step=` becomes a **curve**. Here: validation AUC as the forest grows.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from churn_model import make_pipeline

with mlflow.start_run(run_name="rf_learning_curve"):
    mlflow.log_param("model", "RandomForest")
    for n in [10, 25, 50, 100, 200]:
        rf = make_pipeline(RandomForestClassifier(n_estimators=n, max_depth=8,
                                                  random_state=0, n_jobs=-1)).fit(X_tr, y_tr)
        auc = roc_auc_score(y_va, rf.predict_proba(X_va)[:, 1])
        mlflow.log_metric("val_auc", auc, step=n)
        print(n, round(auc, 4))

10 0.7224
25 0.7243
50 0.7239
100 0.7256
200 0.7269
🏃 View run rf_learning_curve at: http://127.0.0.1:5000/#/experiments/1/runs/40d25aa9c0394038a1d50272baeb69f3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


👉 **In the UI:** open the run → *Model metrics* → `val_auc` is a chart, not a single number.

## Step 5 · Autologging
One line and MLflow captures parameters, training metrics, the model and its signature for you.
Works for scikit-learn, XGBoost, LightGBM, PyTorch, Keras and more.

In [6]:
from xgboost import XGBClassifier

mlflow.sklearn.autolog(log_models=True)
with mlflow.start_run(run_name="xgb_autolog"):
    xgb = make_pipeline(XGBClassifier(max_depth=4, n_estimators=300, learning_rate=0.1,
                                      eval_metric="logloss")).fit(X_tr, y_tr)
    # autolog captures TRAINING metrics; validation metrics are still on you:
    mlflow.log_metric("val_auc", roc_auc_score(y_va, xgb.predict_proba(X_va)[:, 1]))
mlflow.sklearn.autolog(disable=True)

2026/11/04 20:41:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgb_autolog at: http://127.0.0.1:5000/#/experiments/1/runs/1d03ec1f0ea5413fae3318c510bde2af
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


👉 **In the UI:** compare the autolog run with your manual ones. How many parameters did it log?

## Step 6 · Query runs from code
Everything in the UI is queryable. `search_runs` returns a pandas DataFrame.

In [7]:
runs = mlflow.search_runs(experiment_names=["churn-workshop"],
                          filter_string="metrics.val_auc > 0.5",
                          order_by=["metrics.val_auc DESC"])
runs[["tags.mlflow.runName", "metrics.val_auc", "start_time"]].head(10)

,tags.mlflow.runName,metrics.val_auc,start_time
0,logreg_with_artifacts,0.727980,2026-11-04 19:40:51.924+00:00
1,logreg_baseline,0.727980,2026-11-04 19:40:46.557+00:00
2,rf_learning_curve,0.726928,2026-11-04 19:40:55.991+00:00
3,xgb_autolog,0.719911,2026-11-04 19:41:00.131+00:00


## ✍️ Your turn (10 min)
1. Log a new run with a `GradientBoostingClassifier` (in `make_pipeline`). Log `val_auc` and `val_pr_auc`.
2. In the UI, select your 4 best runs and click **Compare**. Which one would you ship, and why?
3. With `search_runs`, list only the runs tagged `stage = baseline`.
   *Hint:* `filter_string="tags.stage = 'baseline'"`.

In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from churn_model import make_pipeline

params = {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05, "random_state": 0}

with mlflow.start_run(run_name="gbc"):
    model = make_pipeline(GradientBoostingClassifier(**params)).fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]

    mlflow.log_params(params)
    mlflow.log_metrics({
        "val_auc": roc_auc_score(y_va, proba),
        "val_pr_auc": average_precision_score(y_va, proba),
    })
    mlflow.set_tags({"stage": "experiment", "model_family": "gradient_boosting"})

    print("val_auc:", roc_auc_score(y_va, proba))

val_auc: 0.7386086564706783
🏃 View run gbc at: http://127.0.0.1:5000/#/experiments/1/runs/adcba352986c4125aa5e71ea1cfbf2ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
